In [2]:
import supervision as sv
import os
import shutil

In [ ]:
# Create directory structure
base_dirs = {
    'coco': 'rfdetr_dataset',
    'yolo': 'yolo_dataset'
}

# Load dataset from COCO format
ds = sv.DetectionDataset.from_coco(
    images_directory_path="Dataset/image", 
    annotations_path="Dataset/_annotations.coco.json"
)

# Split dataset with fixed random seed
ds_train, ds_temp = ds.split(split_ratio=0.7, shuffle=True, random_state=42)
ds_valid, ds_test = ds_temp.split(split_ratio=0.5, shuffle=True, random_state=42)

splits = {
    'train': ds_train,
    'valid': ds_valid,
    'test': ds_test
}

# Save splits in both formats
first_split = True
for split_name, split_data in splits.items():
    # Save COCO format
    split_data.as_coco(
        images_directory_path=f"{base_dirs['coco']}/{split_name}",
        annotations_path=f"{base_dirs['coco']}/{split_name}/_annotations.coco.json"
    )
    
    # Save YOLO format
    split_data.as_yolo(
        images_directory_path=f"{base_dirs['yolo']}/{split_name}/image",
        annotations_directory_path=f"{base_dirs['yolo']}/{split_name}/label",
        data_yaml_path=f"{base_dirs['yolo']}/data.yaml" if first_split else None
    )
    first_split = False

# Copy the original data.yaml to read class names
with open('Dataset/data.yaml', 'r') as f:
    original_yaml = f.read()

# Update the data.yaml with correct paths while preserving class information
with open(f"{base_dirs['yolo']}/data.yaml", 'w') as f:
    f.write(f"""path: ./ # dataset root directory
train: train/image  # train images relative to path
val: valid/image    # validation images relative to path
test: test/image    # test images relative to path

{original_yaml[original_yaml.find('nc:'):]}  # preserve original class information
""")